# ComfyUI + Civitai on Colab
Runtime > Change runtime type > GPU (T4 or better). Add `CIVITAI_TOKEN` in the Colab Secrets panel (key icon) and enable notebook access.

In [ ]:
#@title 1. Clone repo, install ComfyUI, nodes and models
import os
from google.colab import drive, userdata
drive.mount('/content/drive')  # models persist across sessions

REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO'  #@param {type:'string'}
os.environ['CIVITAI_TOKEN'] = userdata.get('CIVITAI_TOKEN')
os.environ['COMFY_DIR'] = '/content/drive/MyDrive/ComfyUI'

!git clone {REPO_URL} /content/repo 2>/dev/null || git -C /content/repo pull
!bash /content/repo/scripts/setup.sh

In [ ]:
#@title 2. Launch ComfyUI (Cloudflare tunnel, prints a trycloudflare.com URL)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, threading, re, time

def tunnel():
    p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stderr=subprocess.PIPE, text=True)
    for line in p.stderr:
        m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
        if m: print('\n>>> OPEN:', m.group(0), '\n')
threading.Thread(target=tunnel, daemon=True).start()
time.sleep(3)
!python $COMFY_DIR/main.py --dont-print-server